# Lost in the Middle: Quick Start Guide

This notebook demonstrates how to use the RAG Research Engine to study and solve the "Lost in the Middle" problem.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from src.core.llm_client import LLMClient
from src.core.document_generator import DocumentGenerator
from src.experiments.dead_zone_mapper import DeadZoneMapperExperiment
from src.experiments.context_restructuring_exp import ContextRestructuringExperiment
from src.strategies.context_restructuring import ContextRestructuringStrategy
from src.strategies.chunked_reading import ChunkedReadingStrategy
from src.strategies.attention_anchoring import AttentionAnchoringStrategy
from src.utils.visualization import ExperimentVisualizer

import matplotlib.pyplot as plt
import numpy as np

print("✓ All imports successful!")

## 1. Understanding the Problem

First, let's create a simple example that demonstrates the "Lost in the Middle" problem.

In [ ]:
# Initialize components
llm_client = LLMClient(
    provider="openai",
    model="gpt-4-turbo-preview",
    temperature=0.0
)

doc_generator = DocumentGenerator()

print("✓ LLM client and document generator initialized")

In [ ]:
# Generate a test document set with needle in the MIDDLE (position 0.5)
needle_doc = doc_generator.create_needle_in_haystack(
    num_documents=20,
    tokens_per_doc=300,
    needle_position=0.5  # Middle position
)

print(f"Generated {len(needle_doc.documents)} documents")
print(f"Needle is in document {needle_doc.needle_doc_index} (position {needle_doc.needle_position:.0%})")
print(f"\nQuestion: {needle_doc.question}")
print(f"Expected answer: {needle_doc.answer}")

In [ ]:
# Build context and test retrieval
context = "\n".join([f"Document {doc.doc_id}:\n{doc.content}\n" for doc in needle_doc.documents])

prompt = f"""Answer the question based ONLY on the provided documents.

{context}

Question: {needle_doc.question}

Answer with ONLY the specific answer:"""

response = llm_client.generate(prompt)

print(f"LLM Response: {response.text}")
print(f"Expected: {needle_doc.answer}")
print(f"\nCorrect: {needle_doc.answer in response.text.upper()}")

## 2. Experiment 1: Dead Zone Mapper

Let's run a quick version of the Dead Zone Mapper to see where attention drops.

In [ ]:
# Configure experiment
config = {
    "llm_provider": "openai",
    "llm_model": "gpt-4-turbo-preview",
    "needle_positions": [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9],
    "num_documents": 20,
    "tokens_per_doc": 300
}

# Create and run experiment (just 3 trials for quick demo)
experiment = DeadZoneMapperExperiment(config)
results = experiment.run(num_trials=3, save=False)

In [ ]:
# Visualize results
visualizer = ExperimentVisualizer()

# Convert results to format needed for visualization
results_dicts = [r.to_dict() for r in results]

fig = visualizer.plot_dead_zone_map(results_dicts)
plt.show()

## 3. Recovery Strategy 1: Context Restructuring

Now let's test if intelligent restructuring can improve retrieval.

In [ ]:
# Create a challenging test case (needle in middle)
test_doc = doc_generator.create_needle_in_haystack(
    num_documents=20,
    tokens_per_doc=300,
    needle_position=0.5
)

restructurer = ContextRestructuringStrategy(llm_client)

# Test different restructuring methods
methods = ["baseline", "relevance", "alternating", "reverse"]
results = {}

for method in methods:
    print(f"\nTesting {method} method...")
    
    if method == "baseline":
        docs = test_doc.documents
    else:
        docs = restructurer.restructure(test_doc.documents, test_doc.question, method=method)
    
    # Build context
    context = "\n".join([f"Document {doc.doc_id}:\n{doc.content}\n" for doc in docs])
    
    # Query
    prompt = f"""Answer based ONLY on these documents.
    
{context}

Question: {test_doc.question}

Answer:"""
    
    response = llm_client.generate(prompt)
    is_correct = test_doc.answer in response.text.upper()
    
    results[method] = {
        "correct": is_correct,
        "response": response.text
    }
    
    print(f"  Result: {'✓ CORRECT' if is_correct else '✗ INCORRECT'}")
    print(f"  Response: {response.text[:100]}...")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for method, result in results.items():
    status = "✓" if result["correct"] else "✗"
    print(f"{status} {method:15s}: {result['correct']}")

## 4. Recovery Strategy 2: Chunked Reading

Test if reading in chunks improves retrieval.

In [ ]:
chunked_reader = ChunkedReadingStrategy(llm_client)

# Test different chunking methods
chunk_methods = ["sequential", "hierarchical", "query_guided"]
chunk_results = {}

for method in chunk_methods:
    print(f"\nTesting {method} chunking...")
    
    answer = chunked_reader.process(
        test_doc.documents,
        test_doc.question,
        method=method,
        chunk_size=5
    )
    
    is_correct = test_doc.answer in answer.upper()
    chunk_results[method] = is_correct
    
    print(f"  Result: {'✓ CORRECT' if is_correct else '✗ INCORRECT'}")
    print(f"  Answer: {answer[:100]}...")

print("\n" + "="*60)
print("CHUNKING STRATEGY RESULTS")
print("="*60)
for method, correct in chunk_results.items():
    status = "✓" if correct else "✗"
    print(f"{status} {method:15s}: {correct}")

## 5. Recovery Strategy 3: Attention Anchoring

Test if formatting and markers help.

In [ ]:
anchorer = AttentionAnchoringStrategy()

# Test different anchoring methods
anchor_methods = ["baseline", "section_markers", "explicit_instructions", "formatting", "combined"]
anchor_results = {}

for method in anchor_methods:
    print(f"\nTesting {method} anchoring...")
    
    # Apply anchoring
    context = anchorer.apply_anchoring(test_doc.documents, method=method, query=test_doc.question)
    
    # Query
    prompt = f"""{context}

Question: {test_doc.question}

Answer:"""
    
    response = llm_client.generate(prompt)
    is_correct = test_doc.answer in response.text.upper()
    
    anchor_results[method] = is_correct
    
    print(f"  Result: {'✓ CORRECT' if is_correct else '✗ INCORRECT'}")

print("\n" + "="*60)
print("ATTENTION ANCHORING RESULTS")
print("="*60)
for method, correct in anchor_results.items():
    status = "✓" if correct else "✗"
    print(f"{status} {method:20s}: {correct}")

## 6. Run Full Experiments

For full experiments with multiple trials and comprehensive analysis, use the command-line interface:

```bash
# Quick test (3 trials)
python ../run_experiments.py dead_zone --quick

# Full experiment (10+ trials)
python ../run_experiments.py dead_zone --trials 15

# All experiments
python ../run_experiments.py all --trials 10
```

Results will be saved to `../results/` with visualizations in `../results/*/visualizations/`

## Key Takeaways

1. **The problem is real** - LLMs consistently miss information in the middle of long contexts

2. **Multiple solutions exist**:
   - Context restructuring (move important info to edges)
   - Chunked reading (process in smaller pieces)
   - Attention anchoring (use formatting to force attention)

3. **Combination approaches work best** - Using multiple strategies together yields the best results

4. **Context matters** - The effectiveness of each strategy depends on:
   - Document length
   - Number of documents
   - Query complexity
   - Model used

## Next Steps

1. Run full experiments with more trials for statistical significance
2. Test with your own documents/datasets
3. Experiment with different models (GPT-4, Claude 3, etc.)
4. Customize strategies for your use case
5. Use findings to optimize your RAG system